# 04 — Streaming: TTFT vs total latency

Notebooks 01–03 measure request-level latency: p50 and p95 of the full round-trip
time. That's the right metric for batch workloads, but the wrong one for anything
user-facing. What drives perceived responsiveness is **TTFT — time to first token**:
how long before the user sees *something*.

This notebook streams a short generative task (2-sentence review summary + sentiment
label) through both providers and compares:

- **TTFT p50 / p95** — latency of the first visible character.
- **Total latency p50 / p95** — time to the last token, same as prior notebooks.
- **Steady-state throughput** — output tokens per second of generation (excluding TTFT).

A generative task is used instead of the earlier classification task because
streaming tool-use / structured-output changes the definition of "first token"
(first `{` byte? first tool_use block?). Plain prose sidesteps that.

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from openai import OpenAI

from src.bench import RESULTS, bench, reset, summarise

load_dotenv()
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
anthropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

OPENAI_MODEL = "gpt-5.4-mini"
ANTHROPIC_MODEL = "claude-haiku-4-5"

import json  # noqa: E402

DATA_PATH = next(p for p in (Path("data/reviews.json"), Path("../data/reviews.json")) if p.exists())
reviews = json.loads(DATA_PATH.read_text())
print(f"Loaded {len(reviews)} reviews")

Loaded 20 reviews


## Prompts — generative summary task

In [2]:
SYSTEM = (
    "You are a product-review analyst. Given a product review, write a concise "
    "two-sentence summary highlighting what the reviewer cared about, then state "
    "the overall sentiment in one word (positive, negative, neutral, or mixed)."
)

USER_TEMPLATE = "Review:\n{review_text}"

## Per-provider streaming functions

For each call we record:
- `ttft_ms` — wall-clock time from request start to the first streamed text chunk.
- `latency_ms` — already captured by the `bench()` context manager.
- `output_tokens`, `input_tokens` — from the post-stream usage object.

In [3]:
def run_openai_stream(review_text: str, rec):
    """Stream an OpenAI chat completion. Sets rec.ttft_ms inside the loop."""
    start = time.perf_counter()
    stream = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        max_completion_tokens=256,  # gpt-5 series requires max_completion_tokens, not max_tokens
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": USER_TEMPLATE.format(review_text=review_text)},
        ],
        stream=True,
        stream_options={"include_usage": True},
    )

    text_parts = []
    usage = None
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
            if rec.ttft_ms == 0.0:
                rec.ttft_ms = (time.perf_counter() - start) * 1000
            text_parts.append(chunk.choices[0].delta.content)
        if getattr(chunk, "usage", None) is not None:
            usage = chunk.usage

    return "".join(text_parts), usage


def run_anthropic_stream(review_text: str, rec):
    """Stream an Anthropic message. Sets rec.ttft_ms inside the loop."""
    start = time.perf_counter()
    with anthropic_client.messages.stream(
        model=ANTHROPIC_MODEL,
        max_tokens=256,
        system=SYSTEM,
        messages=[{"role": "user", "content": USER_TEMPLATE.format(review_text=review_text)}],
    ) as stream:
        text_parts = []
        for text in stream.text_stream:
            if rec.ttft_ms == 0.0:
                rec.ttft_ms = (time.perf_counter() - start) * 1000
            text_parts.append(text)
        final = stream.get_final_message()

    return "".join(text_parts), final.usage

## Warm-up
One discarded call per provider. First-request latency often includes route
resolution / connection setup / model warm-up that would otherwise skew p95.

In [4]:
warm_prompt = "Say hi."
_ = openai_client.chat.completions.create(
    model=OPENAI_MODEL, max_completion_tokens=8,
    messages=[{"role": "user", "content": warm_prompt}],
)
_ = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL, max_tokens=8,
    messages=[{"role": "user", "content": warm_prompt}],
)
print("Warm-up complete")

Warm-up complete


## Run the streaming benchmark

In [5]:
reset()

for row in reviews:
    # --- OpenAI streaming ---
    with bench(f"openai/{OPENAI_MODEL} — streaming", OPENAI_MODEL, "openai") as rec:
        try:
            text, usage = run_openai_stream(row["text"], rec)
            if usage is not None:
                rec.input_tokens = usage.prompt_tokens
                rec.output_tokens = usage.completion_tokens
            rec.ok = bool(text.strip()) and rec.ttft_ms > 0
            rec.extra["text"] = text
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"[:200]

    # --- Anthropic streaming ---
    with bench(f"anthropic/{ANTHROPIC_MODEL} — streaming", ANTHROPIC_MODEL, "anthropic") as rec:
        try:
            text, usage = run_anthropic_stream(row["text"], rec)
            rec.input_tokens = usage.input_tokens
            rec.output_tokens = usage.output_tokens
            rec.ok = bool(text.strip()) and rec.ttft_ms > 0
            rec.extra["text"] = text
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"[:200]

print(f"Collected {len(RESULTS)} records")

Collected 40 records


## Aggregate summary

In [6]:
df = summarise()
df

,label,n,latency_p50_ms,latency_p95_ms,mean_input_tokens,mean_output_tokens,cost_per_1k_usd,ok_rate,ttft_p50_ms,ttft_p95_ms
0,openai/gpt-5.4-mini — streaming,20,1038.2,2100.2,84.9,43.2,0.2581,1.0,620.4,1531.3
1,anthropic/claude-haiku-4-5 — streaming,20,1246.8,1524.0,88.9,49.7,0.3374,1.0,724.8,1022.8


## Steady-state throughput

`tokens/sec = output_tokens / (total_time - ttft)` — the rate of token generation
once the model has started producing output, excluding the time-to-first-token
latency. Useful for estimating streaming bandwidth to a UI.

In [7]:
rows = []
for r in RESULTS:
    gen_time_s = max((r.latency_ms - r.ttft_ms) / 1000.0, 1e-6)
    rows.append({
        "provider": r.provider,
        "model": r.model,
        "ttft_ms": round(r.ttft_ms, 1),
        "total_ms": round(r.latency_ms, 1),
        "gen_time_ms": round(r.latency_ms - r.ttft_ms, 1),
        "output_tokens": r.output_tokens,
        "tokens_per_sec": round(r.output_tokens / gen_time_s, 1),
    })
throughput = pd.DataFrame(rows)
throughput.groupby(["provider", "model"]).agg(
    mean_ttft_ms=("ttft_ms", "mean"),
    mean_total_ms=("total_ms", "mean"),
    mean_gen_time_ms=("gen_time_ms", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_tokens_per_sec=("tokens_per_sec", "mean"),
).round(1)

,,mean_ttft_ms,mean_total_ms,mean_gen_time_ms,mean_output_tokens,mean_tokens_per_sec
provider,model,,,,,
anthropic,claude-haiku-4-5,730.8,1240.2,509.4,49.7,100.9
openai,gpt-5.4-mini,727.8,1094.2,366.4,43.2,127.0


## Per-call detail

Per-review view: which provider showed the first character sooner on each review.
The per-call TTFT is noisy at N=20 but directional — pay attention to the
**p95/worst-case** more than the p50.

In [8]:
by_review = []
oa_records = [r for r in RESULTS if r.provider == "openai"]
an_records = [r for r in RESULTS if r.provider == "anthropic"]
for i, review in enumerate(reviews):
    by_review.append({
        "id": review["id"],
        "text": review["text"][:60] + "…",
        "openai_ttft_ms": round(oa_records[i].ttft_ms, 1),
        "anthropic_ttft_ms": round(an_records[i].ttft_ms, 1),
        "openai_total_ms": round(oa_records[i].latency_ms, 1),
        "anthropic_total_ms": round(an_records[i].latency_ms, 1),
    })
pd.DataFrame(by_review)

,id,text,openai_ttft_ms,anthropic_ttft_ms,openai_total_ms,anthropic_total_ms
0,1,The battery life is incredible — I went three ...,511.3,559.4,940.5,876.3
1,2,"Returned it within a week. Software is buggy, ...",641.2,549.5,1097.9,1227.5
2,3,It's fine. Does what it says on the box. Nothi...,1534.2,997.9,2112.0,1373.4
3,4,Absolutely love it. Noise cancellation is the ...,818.3,612.3,1020.9,1105.9
4,5,"Great sound, terrible battery. Had to charge e...",616.6,578.0,1077.3,1266.1
5,6,Perfect for the price. Runs every game I throw...,966.9,646.1,1346.8,1177.5
6,7,Disappointed. The stitching came undone on the...,524.5,766.6,984.7,1513.6
7,8,"Five stars. Fast delivery, easy setup, exactly...",509.9,842.2,721.8,1182.2
8,9,Mixed feelings. The camera is outstanding but ...,555.6,781.0,955.8,1170.8
9,10,Bought this for travel and it has been a game ...,538.6,913.7,712.2,1386.0
